In [1]:
# ============================================================
# Minimal setup for Step 3A
#
# Upload:
#   1. config.py
#   2. Step 2 zip, e.g. pilot3_step2_chunk_relations.zip
#
# This cell uses paths from /content/pilot_3/config.py.
# ============================================================

import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path
from google.colab import files
from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. Create project structure
# ------------------------------------------------------------

PILOT_ROOT = "/content/pilot_3"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Make pilot_3 importable as a package.
with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# ------------------------------------------------------------
# 2. Upload config.py
# ------------------------------------------------------------

print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = None

for name in uploaded_config.keys():
    # Colab may rename duplicate uploads as:
    #   config (1).py
    #   config (2).py
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

shutil.copy(
    os.path.join("/content", config_upload_name),
    os.path.join(PILOT_ROOT, "config.py"),
)

# ------------------------------------------------------------
# 3. Load config
# ------------------------------------------------------------

import pilot_3.config as cfg
importlib.reload(cfg)

os.makedirs(cfg.STEP2_DIR, exist_ok=True)
os.makedirs(cfg.STEP3A_CANONICAL_DIR, exist_ok=True)
os.makedirs(cfg.STEP3A_DIVERSE_DIR, exist_ok=True)

print("\nLoaded config:")
print("PILOT_ROOT:", cfg.PILOT_ROOT)
print("STEP2_DIR:", cfg.STEP2_DIR)
print("STEP3A_CANONICAL_DIR:", cfg.STEP3A_CANONICAL_DIR)
print("STEP3A_DIVERSE_DIR:", cfg.STEP3A_DIVERSE_DIR)
print("SCENES:", cfg.SCENES)
print("RUN_MODE:", cfg.RUN_MODE)

# ------------------------------------------------------------
# 4. Use Step 2 zip from Google Drive
# ------------------------------------------------------------

step2_zip_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/exp_A_positional_context_6floorplans/settingA_scene_split_mixed_direct_inverse/data_exp_A_positional_context_6floorplans_settingA/pilot4_step2_chunk_relations/pilot3_step2_chunk_relations.zip"
)

print("\nUsing Step 2 zip from Google Drive:")
print(step2_zip_path)

if not step2_zip_path.exists():
    raise FileNotFoundError(f"Step 2 zip not found: {step2_zip_path}")

zip_path = str(step2_zip_path)

# ------------------------------------------------------------
# 5. Unzip Step 2 output
# ------------------------------------------------------------

tmp_extract_dir = "/content/_step2_extract_tmp"

if os.path.exists(tmp_extract_dir):
    shutil.rmtree(tmp_extract_dir)

os.makedirs(tmp_extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(tmp_extract_dir)

# Copy Step 2 relation json files into cfg.STEP2_DIR.
json_files = list(Path(tmp_extract_dir).rglob("*_chunk_relations.json"))

if not json_files:
    raise FileNotFoundError("No *_chunk_relations.json files found in the Step 2 zip.")

for jf in json_files:
    shutil.copy(str(jf), os.path.join(cfg.STEP2_DIR, jf.name))

print("\nSetup complete.")
print("config.py -> /content/pilot_3/config.py")
print("Step 2 relation files ->", cfg.STEP2_DIR)

for name in sorted(os.listdir(cfg.STEP2_DIR)):
    print("-", name)

Mounted at /content/drive
Upload config.py


Saving config_exp_A_positional_context_6floorplans.py to config_exp_A_positional_context_6floorplans.py

Loaded config:
PILOT_ROOT: /content/pilot_3
STEP2_DIR: /content/pilot_3/data/step2_chunk_relations
STEP3A_CANONICAL_DIR: /content/pilot_3/data/step3A_canonical_text_preserve_text_direction
STEP3A_DIVERSE_DIR: /content/pilot_3/data/step3A_diverse_text_preserve_text_direction
SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6']
RUN_MODE: diverse

Using Step 2 zip from Google Drive:
/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/exp_A_positional_context_6floorplans/settingA_scene_split_mixed_direct_inverse/data_exp_A_positional_context_6floorplans_settingA/pilot4_step2_chunk_relations/pilot3_step2_chunk_relations.zip

Setup complete.
config.py -> /content/pilot_3/config.py
Step 2 relation files -> /content/pilot_3/data/step2_chunk_relations
- FloorPlan1_chunk_relations.json
- FloorPlan2_chunk_relations.json
- FloorPlan3_chunk_relation

In [2]:
# ============================================================
# Imports and Step 3A configuration
#
# All configurable parameters are loaded from pilot_3.config.
# Do not redefine Step 3 parameters in this notebook.
# ============================================================

import json
import os
import random
import re
import sys
import hashlib
import shutil
import importlib
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_3.config as cfg
importlib.reload(cfg)

from pilot_3.config import (
    RANDOM_SEED,
    SCENES,

    STEP2_DIR,
    STEP3A_CANONICAL_DIR,
    STEP3A_DIVERSE_DIR,

    EXPERIMENT_NAME,

    MAX_PARAGRAPHS_PER_CLUSTER,

    MIN_RELATIONS_PER_PARAGRAPH,
    TARGET_RELATIONS_PER_PARAGRAPH,
    MAX_RELATIONS_PER_PARAGRAPH,

    USE_PREFERRED_RELATIONS_FIRST,

    ALLOW_NEAR_IN_TEXT,
    MAX_NEAR_SENTENCES_PER_PARAGRAPH,

    ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT,

    INCLUDE_NONE_RELATION_PAIRS,
    NONE_RELATION_LABEL,

    TARGET_OBJECT_COVERAGE_RATIO,
    MAX_OBJECTS_PER_PARAGRAPH,

    NEW_OBJECT_BONUS,
    TWO_NEW_OBJECT_BONUS,

    ALLOW_REDUNDANT_RELATIONS,
    MAX_REDUNDANT_RELATIONS,

    RELATION_SELECTION_JITTER,
    REUSED_RELATION_PENALTY,

    CANONICAL_MODE,
    DIVERSE_MODE,
    RUN_MODE,

    INTRO_TEMPLATES,
    RELATION_TEMPLATES,
    CANONICAL_TEMPLATE,
    TEXT_RELATION_PRIORITY,

    INVERSE_TO_NATURAL,
    NATURAL_INVERSE_SWAP_RELATIONS,
    INVERSE_LABEL_MAP,
)

random.seed(RANDOM_SEED)

os.makedirs(STEP3A_CANONICAL_DIR, exist_ok=True)
os.makedirs(STEP3A_DIVERSE_DIR, exist_ok=True)

print("STEP2_DIR:", STEP2_DIR)
print("STEP3A_CANONICAL_DIR:", STEP3A_CANONICAL_DIR)
print("STEP3A_DIVERSE_DIR:", STEP3A_DIVERSE_DIR)
print("SCENES:", SCENES)
print("RUN_MODE:", RUN_MODE)

print("\nStep 3A config:")
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("MAX_PARAGRAPHS_PER_CLUSTER:", MAX_PARAGRAPHS_PER_CLUSTER)
print("MIN_RELATIONS_PER_PARAGRAPH:", MIN_RELATIONS_PER_PARAGRAPH)
print("TARGET_RELATIONS_PER_PARAGRAPH:", TARGET_RELATIONS_PER_PARAGRAPH)
print("MAX_RELATIONS_PER_PARAGRAPH:", MAX_RELATIONS_PER_PARAGRAPH)
print("USE_PREFERRED_RELATIONS_FIRST:", USE_PREFERRED_RELATIONS_FIRST)
print("ALLOW_NEAR_IN_TEXT:", ALLOW_NEAR_IN_TEXT)
print("MAX_NEAR_SENTENCES_PER_PARAGRAPH:", MAX_NEAR_SENTENCES_PER_PARAGRAPH)
print("ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT:", ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT)
print("INCLUDE_NONE_RELATION_PAIRS:", INCLUDE_NONE_RELATION_PAIRS)
print("NONE_RELATION_LABEL:", NONE_RELATION_LABEL)
print("TARGET_OBJECT_COVERAGE_RATIO:", TARGET_OBJECT_COVERAGE_RATIO)
print("MAX_OBJECTS_PER_PARAGRAPH:", MAX_OBJECTS_PER_PARAGRAPH)
print("ALLOW_REDUNDANT_RELATIONS:", ALLOW_REDUNDANT_RELATIONS)
print("MAX_REDUNDANT_RELATIONS:", MAX_REDUNDANT_RELATIONS)
print("RELATION_SELECTION_JITTER:", RELATION_SELECTION_JITTER)
print("REUSED_RELATION_PENALTY:", REUSED_RELATION_PENALTY)

print("\nStep 2 input files:")
for name in sorted(os.listdir(STEP2_DIR)):
    print("-", name)

STEP2_DIR: /content/pilot_3/data/step2_chunk_relations
STEP3A_CANONICAL_DIR: /content/pilot_3/data/step3A_canonical_text_preserve_text_direction
STEP3A_DIVERSE_DIR: /content/pilot_3/data/step3A_diverse_text_preserve_text_direction
SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6']
RUN_MODE: diverse

Step 3A config:
EXPERIMENT_NAME: A_relation_classification_preserve_text_direction
MAX_PARAGRAPHS_PER_CLUSTER: 4
MIN_RELATIONS_PER_PARAGRAPH: 6
TARGET_RELATIONS_PER_PARAGRAPH: 16
MAX_RELATIONS_PER_PARAGRAPH: 18
USE_PREFERRED_RELATIONS_FIRST: True
ALLOW_NEAR_IN_TEXT: True
MAX_NEAR_SENTENCES_PER_PARAGRAPH: 2
ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT: True
INCLUDE_NONE_RELATION_PAIRS: True
NONE_RELATION_LABEL: none
TARGET_OBJECT_COVERAGE_RATIO: 0.75
MAX_OBJECTS_PER_PARAGRAPH: 12
ALLOW_REDUNDANT_RELATIONS: True
MAX_REDUNDANT_RELATIONS: 8
RELATION_SELECTION_JITTER: 35
REUSED_RELATION_PENALTY: 80

Step 2 input files:
- FloorPlan1_chunk_relations.json
- FloorPlan2

In [3]:
# ============================================================
# Step 3A parameters are loaded from pilot_3.config
# ============================================================

print("Step 3A parameters are controlled by /content/pilot_3/config.py.")
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("RUN_MODE:", RUN_MODE)
print("MAX_PARAGRAPHS_PER_CLUSTER:", MAX_PARAGRAPHS_PER_CLUSTER)
print("MIN_RELATIONS_PER_PARAGRAPH:", MIN_RELATIONS_PER_PARAGRAPH)
print("TARGET_RELATIONS_PER_PARAGRAPH:", TARGET_RELATIONS_PER_PARAGRAPH)
print("MAX_RELATIONS_PER_PARAGRAPH:", MAX_RELATIONS_PER_PARAGRAPH)
print("TARGET_OBJECT_COVERAGE_RATIO:", TARGET_OBJECT_COVERAGE_RATIO)
print("MAX_OBJECTS_PER_PARAGRAPH:", MAX_OBJECTS_PER_PARAGRAPH)

Step 3A parameters are controlled by /content/pilot_3/config.py.
EXPERIMENT_NAME: A_relation_classification_preserve_text_direction
RUN_MODE: diverse
MAX_PARAGRAPHS_PER_CLUSTER: 4
MIN_RELATIONS_PER_PARAGRAPH: 6
TARGET_RELATIONS_PER_PARAGRAPH: 16
MAX_RELATIONS_PER_PARAGRAPH: 18
TARGET_OBJECT_COVERAGE_RATIO: 0.75
MAX_OBJECTS_PER_PARAGRAPH: 12


In [4]:
# ============================================================
# Relation templates are loaded from pilot_3.config
# ============================================================

print("Loaded relation templates from config:")

for relation, templates in RELATION_TEMPLATES.items():
    print(f"- {relation}: {len(templates)} templates; canonical = {templates[0]}")

print("\nLoaded relation priority from config:")
print(TEXT_RELATION_PRIORITY)

Loaded relation templates from config:
- in: 10 templates; canonical = {subj} is in {obj}.
- on: 9 templates; canonical = {subj} is on {obj}.
- left_of: 6 templates; canonical = {subj} is to the left of {obj}.
- right_of: 6 templates; canonical = {subj} is to the right of {obj}.
- above: 7 templates; canonical = {subj} is above {obj}.
- below: 7 templates; canonical = {subj} is below {obj}.
- near: 8 templates; canonical = {subj} is near {obj}.
- supports: 5 templates; canonical = {subj} supports {obj}.
- contains: 6 templates; canonical = {subj} contains {obj}.

Loaded relation priority from config:
{'on': 100, 'in': 100, 'above': 90, 'below': 90, 'left_of': 80, 'right_of': 80, 'near': 50, 'supports': 40, 'contains': 40}


In [5]:
def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: str, data: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def stable_sort_relations(relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return sorted(
        relations,
        key=lambda r: (
            r.get("scene", ""),
            r.get("chunk_id", ""),
            r.get("cluster_id", ""),
            r.get("relation", ""),
            r.get("subject_type", ""),
            r.get("object_type", ""),
            r.get("subject_id", ""),
            r.get("object_id", ""),
        )
    )


def stable_hash_int(text: str) -> int:
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest, 16)


print("IO helpers loaded.")

IO helpers loaded.


In [6]:
def camel_to_snake(name: str) -> str:
    if not name:
        return "object"

    s1 = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    s2 = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s1)
    return s2.lower()


def build_alias_map_from_relations(
    relations: List[Dict[str, Any]]
) -> Dict[str, Dict[str, Any]]:
    object_entries = {}

    for rel in relations:
        sid = rel["subject_id"]
        oid = rel["object_id"]

        if sid not in object_entries:
            object_entries[sid] = {
                "object_id": sid,
                "object_type": rel["subject_type"],
            }

        if oid not in object_entries:
            object_entries[oid] = {
                "object_id": oid,
                "object_type": rel["object_type"],
            }

    type_counter = defaultdict(int)
    alias_map = {}

    sorted_items = sorted(
        object_entries.items(),
        key=lambda item: (item[1]["object_type"], item[0])
    )

    for object_id, info in sorted_items:
        base = camel_to_snake(info["object_type"])
        type_counter[base] += 1

        object_uid = f"{base}_{type_counter[base]}"

        alias_map[object_id] = {
            "object_uid": object_uid,
            "object_id": object_id,
            "object_type": info["object_type"],
            "mention_text": object_uid,
        }

    return alias_map


def get_uid(alias_map: Dict[str, Dict[str, Any]], object_id: str) -> str:
    return alias_map[object_id]["object_uid"]


print("Alias helpers loaded.")

Alias helpers loaded.


In [7]:
# ============================================================
# Relation normalization helpers
#
# INVERSE_TO_NATURAL and NATURAL_INVERSE_SWAP_RELATIONS
# are loaded from pilot_3.config.
# ============================================================

def relation_priority_for_text(rel: Dict[str, Any]) -> int:
    return TEXT_RELATION_PRIORITY.get(rel.get("relation", ""), 0)


def normalize_relation_for_text(rel: Dict[str, Any]) -> Dict[str, Any]:
    r = rel["relation"]

    if ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT:
        return {
            "subject_id": rel["subject_id"],
            "subject_type": rel["subject_type"],
            "relation": r,
            "object_id": rel["object_id"],
            "object_type": rel["object_type"],
            "source_relation": rel,
            "was_swapped_for_text": False,
        }

    if r in NATURAL_INVERSE_SWAP_RELATIONS:
        return {
            "subject_id": rel["object_id"],
            "subject_type": rel["object_type"],
            "relation": INVERSE_TO_NATURAL[r],
            "object_id": rel["subject_id"],
            "object_type": rel["subject_type"],
            "source_relation": rel,
            "was_swapped_for_text": True,
        }

    return {
        "subject_id": rel["subject_id"],
        "subject_type": rel["subject_type"],
        "relation": r,
        "object_id": rel["object_id"],
        "object_type": rel["object_type"],
        "source_relation": rel,
        "was_swapped_for_text": False,
    }


def deduplicate_text_relations(text_relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    output = []

    for tr in text_relations:
        key = (tr["subject_id"], tr["relation"], tr["object_id"])

        if key in seen:
            continue

        seen.add(key)
        output.append(tr)

    return output


print("Relation normalization helpers loaded.")

Relation normalization helpers loaded.


In [8]:
def select_template_for_relation(
    relation: str,
    mode: str,
    paragraph_id: str,
    relation_index: int,
    subject_uid: str,
    object_uid: str,
) -> Tuple[str, int]:
    templates = RELATION_TEMPLATES.get(
        relation,
        [f"{{subj}} is spatially related to {{obj}} by {relation}."]
    )

    if mode == CANONICAL_MODE:
        return templates[0], 0

    if mode == DIVERSE_MODE:
        # Deterministic pseudo-random choice.
        # This makes reruns stable while producing varied descriptions.
        key = f"{RANDOM_SEED}|{paragraph_id}|{relation_index}|{relation}|{subject_uid}|{object_uid}"
        idx = stable_hash_int(key) % len(templates)
        return templates[idx], idx

    raise ValueError(f"Unknown generation mode: {mode}")


def sentence_from_text_relation(
    text_rel: Dict[str, Any],
    alias_map: Dict[str, Dict[str, Any]],
    mode: str,
    paragraph_id: str,
    relation_index: int,
) -> Dict[str, Any]:
    s = get_uid(alias_map, text_rel["subject_id"])
    o = get_uid(alias_map, text_rel["object_id"])
    r = text_rel["relation"]

    template, template_index = select_template_for_relation(
        relation=r,
        mode=mode,
        paragraph_id=paragraph_id,
        relation_index=relation_index,
        subject_uid=s,
        object_uid=o,
    )

    sentence = template.format(subj=s, obj=o)

    return {
        "sentence": sentence,
        "template": template,
        "template_index": template_index,
        "generation_mode": mode,
    }


print("Template selection and sentence generation loaded.")

Template selection and sentence generation loaded.


In [9]:
# ============================================================
# Coverage-aware relation selection
# Relation-count-target version
# Replace the old filter_relations_for_text cell with this cell.
# ============================================================

def get_relation_object_ids(rel: Dict[str, Any]) -> Tuple[str, str]:
    """
    Return the two object ids used by a relation.
    """
    return rel["subject_id"], rel["object_id"]


def get_cluster_object_ids(cluster: Dict[str, Any]) -> List[str]:
    """
    Get object ids from the cluster itself.

    Step 2 cluster usually preserves object_ids or objects.
    This helper is robust to both formats.
    """
    if "object_ids" in cluster and cluster["object_ids"]:
        return list(cluster["object_ids"])

    if "objects" in cluster and cluster["objects"]:
        return [
            obj["objectId"]
            for obj in cluster["objects"]
            if "objectId" in obj
        ]

    # Fallback: infer from relations
    ids = set()
    for rel in cluster.get("relations", []):
        ids.add(rel["subject_id"])
        ids.add(rel["object_id"])

    return sorted(ids)


def relation_key(rel: Dict[str, Any]) -> Tuple[str, str, str]:
    """
    Unique directed relation key.
    """
    return (
        rel["subject_id"],
        rel["relation"],
        rel["object_id"],
    )


def relation_base_score(rel: Dict[str, Any]) -> int:
    """
    Base quality score of a relation before object-coverage reward.
    """
    score = relation_priority_for_text(rel)

    if rel.get("preferred_for_paragraph", False):
        score += 40

    if rel.get("relation_family") in {"support", "containment", "vertical"}:
        score += 20

    if rel.get("relation") == "near":
        score -= 20

    return score


def relation_variant_jitter(
    rel: Dict[str, Any],
    cluster: Dict[str, Any],
    paragraph_index: int,
) -> int:
    """
    Stable pseudo-random jitter.

    Different paragraph_index leads to slightly different relation selection,
    while keeping the result reproducible under RANDOM_SEED.

    paragraph_index == 0 keeps the original deterministic ranking.
    """
    if paragraph_index == 0:
        return 0

    cluster_identity = (
        cluster.get("cluster_id")
        or cluster.get("chunk_id")
        or "|".join(get_cluster_object_ids(cluster))
    )

    key = (
        f"{RANDOM_SEED}|"
        f"{cluster_identity}|"
        f"{paragraph_index}|"
        f"{rel['subject_id']}|"
        f"{rel['relation']}|"
        f"{rel['object_id']}"
    )

    span = 2 * RELATION_SELECTION_JITTER + 1
    return stable_hash_int(key) % span - RELATION_SELECTION_JITTER


def filter_candidate_relations_for_text(relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    First-stage filtering: remove relations that should not be verbalized.
    """
    candidates = []

    for rel in relations:
        if not rel.get("verbalizable", True):
            continue

        if not rel.get("candidate_for_text", True):
            continue

        if rel.get("relation") == "near" and not ALLOW_NEAR_IN_TEXT:
            continue

        candidates.append(rel)

    # Prefer Step 2 preferred relations, but do not discard all non-preferred relations
    # because non-preferred relations may introduce additional objects.
    if USE_PREFERRED_RELATIONS_FIRST:
        preferred = [
            rel for rel in candidates
            if rel.get("preferred_for_paragraph", False)
        ]

        if len(preferred) >= MIN_RELATIONS_PER_PARAGRAPH:
            preferred_keys = {
                relation_key(r)
                for r in preferred
            }

            non_preferred = [
                r for r in candidates
                if relation_key(r) not in preferred_keys
            ]

            candidates = preferred + non_preferred

    return candidates


def selection_score_for_relation(
    rel: Dict[str, Any],
    cluster: Dict[str, Any],
    paragraph_index: int,
    avoid_relation_keys: set,
) -> int:
    """
    Relation quality score with paragraph-specific variation
    and repeated-relation penalty.
    """
    key = relation_key(rel)

    score = relation_base_score(rel)

    if key in avoid_relation_keys:
        score -= REUSED_RELATION_PENALTY

    score += relation_variant_jitter(
        rel=rel,
        cluster=cluster,
        paragraph_index=paragraph_index,
    )

    return score


def select_relations_for_object_coverage(
    relations: List[Dict[str, Any]],
    cluster: Dict[str, Any],
    paragraph_index: int = 0,
    avoid_relation_keys: Optional[set] = None,
) -> List[Dict[str, Any]]:
    """
    Select relations for paragraph generation.

    Goal:
        generate enough explicit relation sentences for probing,
        while still keeping reasonable object coverage.

    New behavior:
        The selector no longer stops immediately when object coverage is reached.
        It keeps adding relations until TARGET_RELATIONS_PER_PARAGRAPH is reached,
        or until MAX_RELATIONS_PER_PARAGRAPH / constraints prevent further selection.
    """
    if avoid_relation_keys is None:
        avoid_relation_keys = set()

    candidates = filter_candidate_relations_for_text(relations)

    if not candidates:
        return []

    cluster_object_ids = get_cluster_object_ids(cluster)
    cluster_object_id_set = set(cluster_object_ids)

    target_num_objects = max(
        MIN_RELATIONS_PER_PARAGRAPH,
        int(round(len(cluster_object_id_set) * TARGET_OBJECT_COVERAGE_RATIO))
    )

    if MAX_OBJECTS_PER_PARAGRAPH is not None:
        target_num_objects = min(target_num_objects, MAX_OBJECTS_PER_PARAGRAPH)

    target_num_relations = min(
        TARGET_RELATIONS_PER_PARAGRAPH,
        MAX_RELATIONS_PER_PARAGRAPH,
    )

    selected = []
    selected_keys = set()
    covered_objects = set()
    near_count = 0
    redundant_count = 0

    remaining = sorted(
        candidates,
        key=lambda r: (
            -selection_score_for_relation(
                rel=r,
                cluster=cluster,
                paragraph_index=paragraph_index,
                avoid_relation_keys=avoid_relation_keys,
            ),
            r.get("relation", ""),
            r.get("subject_type", ""),
            r.get("object_type", ""),
            r.get("subject_id", ""),
            r.get("object_id", ""),
        )
    )

    while remaining and len(selected) < MAX_RELATIONS_PER_PARAGRAPH:
        best_idx = None
        best_score = None

        for idx, rel in enumerate(remaining):
            s, o = get_relation_object_ids(rel)
            key = relation_key(rel)

            if key in selected_keys:
                continue

            if rel.get("relation") == "near" and near_count >= MAX_NEAR_SENTENCES_PER_PARAGRAPH:
                continue

            rel_objects = {s, o}
            new_objects = rel_objects - covered_objects
            num_new = len(new_objects)

            if MAX_OBJECTS_PER_PARAGRAPH is not None:
                possible_object_count = len(covered_objects | rel_objects)
                if possible_object_count > MAX_OBJECTS_PER_PARAGRAPH:
                    continue

            if num_new == 0:
                if not ALLOW_REDUNDANT_RELATIONS:
                    continue

                if redundant_count >= MAX_REDUNDANT_RELATIONS:
                    continue

            score = selection_score_for_relation(
                rel=rel,
                cluster=cluster,
                paragraph_index=paragraph_index,
                avoid_relation_keys=avoid_relation_keys,
            )

            # Before reaching object coverage, reward adding new objects.
            if len(covered_objects) < target_num_objects:
                if num_new == 1:
                    score += NEW_OBJECT_BONUS
                elif num_new == 2:
                    score += NEW_OBJECT_BONUS + TWO_NEW_OBJECT_BONUS
            else:
                # Once coverage is satisfied, prefer adding more relation sentences
                # among already mentioned objects, so explicit relation count increases
                # without exploding object_mentions.
                if num_new == 0:
                    score += 20
                elif num_new == 1:
                    score += 5
                elif num_new == 2:
                    score += 0

            if s in cluster_object_id_set:
                score += 5
            if o in cluster_object_id_set:
                score += 5

            tie_break = (
                score,
                relation_base_score(rel),
                num_new,
                -idx,
            )

            if best_score is None or tie_break > best_score:
                best_score = tie_break
                best_idx = idx

        if best_idx is None:
            break

        chosen = remaining.pop(best_idx)
        s, o = get_relation_object_ids(chosen)
        key = relation_key(chosen)

        before = set(covered_objects)

        selected.append(chosen)
        selected_keys.add(key)
        covered_objects.update([s, o])

        if chosen.get("relation") == "near":
            near_count += 1

        if len(covered_objects - before) == 0:
            redundant_count += 1

        # New stopping logic:
        # Do not stop merely because object coverage is satisfied.
        # Stop only when both object coverage and target relation count are satisfied.
        coverage_satisfied = len(covered_objects) >= target_num_objects
        target_relation_count_satisfied = len(selected) >= target_num_relations

        if coverage_satisfied and target_relation_count_satisfied:
            break

        if len(selected) >= MAX_RELATIONS_PER_PARAGRAPH:
            break

    # Fallback: if relation count is still too small, fill by quality.
    if len(selected) < MIN_RELATIONS_PER_PARAGRAPH:
        selected_keys = {
            relation_key(r)
            for r in selected
        }

        fallback = sorted(
            candidates,
            key=lambda r: (
                -selection_score_for_relation(
                    rel=r,
                    cluster=cluster,
                    paragraph_index=paragraph_index,
                    avoid_relation_keys=avoid_relation_keys,
                ),
                r.get("relation", ""),
                r.get("subject_type", ""),
                r.get("object_type", ""),
                r.get("subject_id", ""),
                r.get("object_id", ""),
            )
        )

        for rel in fallback:
            if len(selected) >= MIN_RELATIONS_PER_PARAGRAPH:
                break

            key = relation_key(rel)

            if key in selected_keys:
                continue

            if rel.get("relation") == "near" and near_count >= MAX_NEAR_SENTENCES_PER_PARAGRAPH:
                continue

            s, o = get_relation_object_ids(rel)

            if MAX_OBJECTS_PER_PARAGRAPH is not None:
                possible_object_count = len(covered_objects | {s, o})
                if possible_object_count > MAX_OBJECTS_PER_PARAGRAPH:
                    continue

            selected.append(rel)
            selected_keys.add(key)
            covered_objects.update([s, o])

            if rel.get("relation") == "near":
                near_count += 1

    return selected


def filter_relations_for_text(
    relations: List[Dict[str, Any]],
    cluster: Optional[Dict[str, Any]] = None,
    paragraph_index: int = 0,
    avoid_relation_keys: Optional[set] = None,
) -> List[Dict[str, Any]]:
    """
    Wrapper used by paragraph generation.

    New behavior:
        If cluster is provided, use coverage-aware selection with paragraph-level variation.
        Otherwise fall back to priority-based selection.
    """
    if cluster is not None:
        return select_relations_for_object_coverage(
            relations=relations,
            cluster=cluster,
            paragraph_index=paragraph_index,
            avoid_relation_keys=avoid_relation_keys,
        )

    candidates = filter_candidate_relations_for_text(relations)

    candidates = sorted(
        candidates,
        key=lambda r: (
            -relation_priority_for_text(r),
            r.get("relation", ""),
            r.get("subject_type", ""),
            r.get("object_type", ""),
            r.get("subject_id", ""),
            r.get("object_id", ""),
        )
    )

    selected = []
    near_count = 0

    for rel in candidates:
        if len(selected) >= MAX_RELATIONS_PER_PARAGRAPH:
            break

        if rel.get("relation") == "near":
            if near_count >= MAX_NEAR_SENTENCES_PER_PARAGRAPH:
                continue
            near_count += 1

        selected.append(rel)

    return selected


print("Coverage-aware relation selection helpers loaded.")

Coverage-aware relation selection helpers loaded.


In [10]:
# ============================================================
# Pair target helpers
#
# INVERSE_LABEL_MAP is loaded from pilot_3.config.
# ============================================================

def build_relation_label_index(
    relations: List[Dict[str, Any]]
) -> Dict[Tuple[str, str], Dict[str, Any]]:
    index = {}

    for rel in stable_sort_relations(relations):
        s = rel["subject_id"]
        o = rel["object_id"]
        r = rel["relation"]

        key = (s, o)

        if key not in index:
            index[key] = {
                "relation_label": r,
                "relation_record": rel,
                "relation_source": "direct",
            }

    current_items = list(index.items())

    for (s, o), info in current_items:
        r = info["relation_label"]

        if r not in INVERSE_LABEL_MAP:
            continue

        inverse_r = INVERSE_LABEL_MAP[r]
        reverse_key = (o, s)

        if reverse_key not in index:
            index[reverse_key] = {
                "relation_label": inverse_r,
                "relation_record": info["relation_record"],
                "relation_source": "inferred_inverse",
            }

    return index


def build_pair_targets_for_paragraph(
    paragraph_id: str,
    scene: str,
    chunk_id: str,
    cluster_id: str,
    alias_map: Dict[str, Dict[str, Any]],
    all_cluster_relations: List[Dict[str, Any]],
    explicit_text_relations: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    object_ids = sorted(alias_map.keys())
    relation_index = build_relation_label_index(all_cluster_relations)

    explicit_unordered_pairs = set()
    explicit_directed_pairs = set()

    for tr in explicit_text_relations:
        s = tr["subject_id"]
        o = tr["object_id"]

        explicit_unordered_pairs.add(tuple(sorted([s, o])))
        explicit_directed_pairs.add((s, o))

    pair_targets = []

    for subject_id in object_ids:
        for object_id in object_ids:
            if subject_id == object_id:
                continue

            key = (subject_id, object_id)
            info = relation_index.get(key)

            if info is None:
                if not INCLUDE_NONE_RELATION_PAIRS:
                    continue

                relation_label = NONE_RELATION_LABEL
                relation_source = "none"
                has_relation_label = False
                geometry_label = None
                source_relation_record = None

            else:
                relation_label = info["relation_label"]
                relation_source = info["relation_source"]
                has_relation_label = True
                source_relation_record = info["relation_record"]
                geometry_label = source_relation_record.get("geometry_label")

            unordered = tuple(sorted([subject_id, object_id]))

            if key in explicit_directed_pairs:
                pair_evidence_type = "explicit_direct"
            elif unordered in explicit_unordered_pairs:
                pair_evidence_type = "explicit_inverse_or_same_sentence_pair"
            elif has_relation_label:
                pair_evidence_type = "implicit_labeled"
            else:
                pair_evidence_type = "implicit_none"

            s_info = alias_map[subject_id]
            o_info = alias_map[object_id]

            pair_targets.append({
                "paragraph_id": paragraph_id,
                "scene": scene,
                "chunk_id": chunk_id,
                "cluster_id": cluster_id,

                "subject_uid": s_info["object_uid"],
                "subject_id": subject_id,
                "subject_type": s_info["object_type"],

                "object_uid": o_info["object_uid"],
                "object_id": object_id,
                "object_type": o_info["object_type"],

                "relation_label": relation_label,
                "has_relation_label": has_relation_label,
                "relation_source": relation_source,
                "pair_evidence_type": pair_evidence_type,

                "geometry_label": geometry_label,

                "source_relation_summary": None if source_relation_record is None else {
                    "subject_id": source_relation_record.get("subject_id"),
                    "relation": source_relation_record.get("relation"),
                    "object_id": source_relation_record.get("object_id"),
                    "relation_family": source_relation_record.get("relation_family"),
                }
            })

    return pair_targets


print("Pair target helpers loaded.")

Pair target helpers loaded.


In [11]:
# ============================================================
# Build one paragraph record from one cluster
# Coverage-aware version
#
# INTRO_TEMPLATES is loaded from pilot_3.config.
# ============================================================

def make_paragraph_intro(
    scene: str,
    chunk_id: str,
    cluster_id: str,
    mode: str,
    paragraph_index: int = 0,
) -> str:
    """
    Make a stable but paragraph-specific intro.
    """
    key = f"{RANDOM_SEED}|{mode}|{scene}|{chunk_id}|{cluster_id}|{paragraph_index}"
    idx = stable_hash_int(key) % len(INTRO_TEMPLATES)
    return INTRO_TEMPLATES[idx]


def build_paragraph_record_from_cluster(
    scene: str,
    chunk: Dict[str, Any],
    cluster: Dict[str, Any],
    mode: str,
    paragraph_index: int = 0,
    avoid_relation_keys: Optional[set] = None,
) -> Optional[Dict[str, Any]]:
    cluster_relations = cluster.get("relations", []) or []

    if not cluster_relations:
        return None

    if avoid_relation_keys is None:
        avoid_relation_keys = set()

    selected_relations = filter_relations_for_text(
        cluster_relations,
        cluster=cluster,
        paragraph_index=paragraph_index,
        avoid_relation_keys=avoid_relation_keys,
    )

    if len(selected_relations) < MIN_RELATIONS_PER_PARAGRAPH:
        return None

    text_relations = [
        normalize_relation_for_text(rel)
        for rel in selected_relations
    ]
    text_relations = deduplicate_text_relations(text_relations)

    if len(text_relations) < MIN_RELATIONS_PER_PARAGRAPH:
        return None

    alias_map = build_alias_map_from_relations(text_relations)

    if MAX_OBJECTS_PER_PARAGRAPH is not None:
        if len(alias_map) > MAX_OBJECTS_PER_PARAGRAPH:
            return None

    paragraph_id = f"{scene}_{cluster['cluster_id']}_A_{mode}_{paragraph_index}"

    sentence_records = []
    sentences = []

    for relation_index, tr in enumerate(text_relations):
        sent_info = sentence_from_text_relation(
            text_rel=tr,
            alias_map=alias_map,
            mode=mode,
            paragraph_id=paragraph_id,
            relation_index=relation_index,
        )

        sentences.append(sent_info["sentence"])

        sentence_records.append({
            "sentence": sent_info["sentence"],
            "template": sent_info["template"],
            "template_index": sent_info["template_index"],
            "generation_mode": mode,

            "subject_uid": get_uid(alias_map, tr["subject_id"]),
            "subject_id": tr["subject_id"],
            "subject_type": tr["subject_type"],

            "relation": tr["relation"],

            "object_uid": get_uid(alias_map, tr["object_id"]),
            "object_id": tr["object_id"],
            "object_type": tr["object_type"],

            "was_swapped_for_text": tr["was_swapped_for_text"],
            "source_relation": {
                "subject_id": tr["source_relation"].get("subject_id"),
                "relation": tr["source_relation"].get("relation"),
                "object_id": tr["source_relation"].get("object_id"),
                "relation_family": tr["source_relation"].get("relation_family"),
            },
        })

    intro = make_paragraph_intro(
        scene=scene,
        chunk_id=chunk["chunk_id"],
        cluster_id=cluster["cluster_id"],
        mode=mode,
        paragraph_index=paragraph_index,
    )

    text = intro + " " + " ".join(sentences)

    object_mentions = list(alias_map.values())
    object_mentions = sorted(object_mentions, key=lambda x: x["object_uid"])

    pair_targets = build_pair_targets_for_paragraph(
        paragraph_id=paragraph_id,
        scene=scene,
        chunk_id=chunk["chunk_id"],
        cluster_id=cluster["cluster_id"],
        alias_map=alias_map,
        all_cluster_relations=cluster_relations,
        explicit_text_relations=text_relations,
    )

    label_counts = dict(Counter(pt["relation_label"] for pt in pair_targets))
    evidence_counts = dict(Counter(pt["pair_evidence_type"] for pt in pair_targets))

    source_cluster_object_ids = get_cluster_object_ids(cluster)
    source_cluster_num_objects = len(source_cluster_object_ids)
    num_object_mentions = len(object_mentions)

    if source_cluster_num_objects > 0:
        object_coverage_ratio = num_object_mentions / source_cluster_num_objects
    else:
        object_coverage_ratio = None

    selected_relation_keys = [
        relation_key(rel)
        for rel in selected_relations
    ]

    text_relation_keys = [
        (
            tr["subject_id"],
            tr["relation"],
            tr["object_id"],
        )
        for tr in text_relations
    ]

    return {
        "paragraph_id": paragraph_id,
        "experiment": EXPERIMENT_NAME,
        "generation_mode": mode,

        "scene": scene,
        "chunk_id": chunk["chunk_id"],
        "cluster_id": cluster["cluster_id"],
        "source_chunk_id": cluster.get("source_chunk_id", chunk["chunk_id"]),
        "grid_i": cluster.get("grid_i"),
        "grid_j": cluster.get("grid_j"),

        "paragraph_index_within_cluster": paragraph_index,

        "text": text,

        "object_mentions": object_mentions,
        "explicit_relations_in_text": sentence_records,
        "pair_targets": pair_targets,

        "diagnostics": {
            "num_text_relations": len(text_relations),
            "target_relations_per_paragraph": TARGET_RELATIONS_PER_PARAGRAPH,
            "max_relations_per_paragraph": MAX_RELATIONS_PER_PARAGRAPH,
            "min_relations_per_paragraph": MIN_RELATIONS_PER_PARAGRAPH,

            "num_object_mentions": num_object_mentions,
            "num_pair_targets": len(pair_targets),

            "source_cluster_num_objects": source_cluster_num_objects,
            "target_object_coverage_ratio": TARGET_OBJECT_COVERAGE_RATIO,
            "object_coverage_ratio": object_coverage_ratio,
            "max_objects_per_paragraph": MAX_OBJECTS_PER_PARAGRAPH,

            "paragraph_index_within_cluster": paragraph_index,
            "num_avoid_relation_keys_received": len(avoid_relation_keys),
            "selected_relation_keys": selected_relation_keys,
            "text_relation_keys": text_relation_keys,

            "pair_label_counts": label_counts,
            "pair_evidence_counts": evidence_counts,
            "text_length_chars": len(text),
            "text_length_words_approx": len(text.split()),
            "source_cluster_num_relations": cluster.get("num_relations"),
            "source_cluster_relation_counts": cluster.get("relation_counts"),
            "source_cluster_preferred_relation_counts": cluster.get("preferred_relation_counts"),
        }
    }


print("Coverage-aware cluster paragraph builder loaded.")

Coverage-aware cluster paragraph builder loaded.


In [12]:
# ============================================================
# Build Step 3A output for one scene
# Replace the old build_scene_step3a cell with this cell.
# ============================================================

def build_scene_step3a(scene_data: Dict[str, Any], mode: str) -> Dict[str, Any]:
    """
    Build Step 3A output for one scene under one selected generation mode.

    mode:
        "canonical" or "diverse"
    """
    scene = scene_data["scene"]

    paragraph_records = []

    for chunk in scene_data.get("chunks", []):
        for cluster in chunk.get("clusters", []):

            # Track explicit relations already selected for this cluster.
            # Later paragraphs from the same cluster will avoid reusing them.
            used_relation_keys_for_cluster = set()

            for paragraph_index in range(MAX_PARAGRAPHS_PER_CLUSTER):
                record = build_paragraph_record_from_cluster(
                    scene=scene,
                    chunk=chunk,
                    cluster=cluster,
                    mode=mode,
                    paragraph_index=paragraph_index,
                    avoid_relation_keys=used_relation_keys_for_cluster,
                )

                if record is not None:
                    paragraph_records.append(record)

                    # Mark text-facing explicit relations as used.
                    for rel in record["explicit_relations_in_text"]:
                        used_relation_keys_for_cluster.add(
                            (
                                rel["subject_id"],
                                rel["relation"],
                                rel["object_id"],
                            )
                        )

                    # Also mark source relation keys when available.
                    # This is useful because normalize_relation_for_text may convert
                    # supports -> on or contains -> in.
                    for rel in record["explicit_relations_in_text"]:
                        src = rel.get("source_relation", {})
                        if src.get("subject_id") and src.get("relation") and src.get("object_id"):
                            used_relation_keys_for_cluster.add(
                                (
                                    src["subject_id"],
                                    src["relation"],
                                    src["object_id"],
                                )
                            )

    all_pair_targets = []
    for rec in paragraph_records:
        all_pair_targets.extend(rec["pair_targets"])

    paragraph_relation_counts = dict(
        Counter(
            rel["relation"]
            for rec in paragraph_records
            for rel in rec["explicit_relations_in_text"]
        )
    )

    pair_label_counts = dict(
        Counter(
            pt["relation_label"]
            for pt in all_pair_targets
        )
    )

    pair_evidence_counts = dict(
        Counter(
            pt["pair_evidence_type"]
            for pt in all_pair_targets
        )
    )

    paragraph_text_relation_counts = [
        rec["diagnostics"]["num_text_relations"]
        for rec in paragraph_records
    ]

    return {
        "scene": scene,
        "experiment": EXPERIMENT_NAME,
        "generation_mode": mode,

        "source_step2_summary": {
            "num_chunks": scene_data.get("num_chunks"),
            "num_relations": scene_data.get("num_relations"),
            "relation_counts": scene_data.get("relation_counts"),
            "preferred_relation_counts": scene_data.get("preferred_relation_counts"),
            "num_geometry_pairs": scene_data.get("num_geometry_pairs"),
        },

        "step3a_config": {
            "generation_mode": mode,
            "max_paragraphs_per_cluster": MAX_PARAGRAPHS_PER_CLUSTER,
            "max_relations_per_paragraph": MAX_RELATIONS_PER_PARAGRAPH,
            "min_relations_per_paragraph": MIN_RELATIONS_PER_PARAGRAPH,
            "target_relations_per_paragraph": TARGET_RELATIONS_PER_PARAGRAPH,
            "use_preferred_relations_first": USE_PREFERRED_RELATIONS_FIRST,
            "allow_near_in_text": ALLOW_NEAR_IN_TEXT,
            "max_near_sentences_per_paragraph": MAX_NEAR_SENTENCES_PER_PARAGRAPH,
            "allow_inverse_topological_in_text": ALLOW_INVERSE_TOPOLOGICAL_IN_TEXT,
            "include_none_relation_pairs": INCLUDE_NONE_RELATION_PAIRS,
            "none_relation_label": NONE_RELATION_LABEL,

            "target_object_coverage_ratio": TARGET_OBJECT_COVERAGE_RATIO,
            "max_objects_per_paragraph": MAX_OBJECTS_PER_PARAGRAPH,
            "new_object_bonus": NEW_OBJECT_BONUS,
            "two_new_object_bonus": TWO_NEW_OBJECT_BONUS,
            "allow_redundant_relations": ALLOW_REDUNDANT_RELATIONS,
            "max_redundant_relations": MAX_REDUNDANT_RELATIONS,

            "relation_selection_jitter": RELATION_SELECTION_JITTER,
            "reused_relation_penalty": REUSED_RELATION_PENALTY,

            "random_seed": RANDOM_SEED,
        },

        "num_paragraphs": len(paragraph_records),
        "num_pair_targets": len(all_pair_targets),

        "paragraph_relation_counts": paragraph_relation_counts,
        "pair_label_counts": pair_label_counts,
        "pair_evidence_counts": pair_evidence_counts,

        "paragraph_text_relation_count_summary": {
            "min": min(paragraph_text_relation_counts) if paragraph_text_relation_counts else None,
            "max": max(paragraph_text_relation_counts) if paragraph_text_relation_counts else None,
            "mean": (
                sum(paragraph_text_relation_counts) / len(paragraph_text_relation_counts)
                if paragraph_text_relation_counts else None
            ),
            "counts": paragraph_text_relation_counts,
        },

        "paragraphs": paragraph_records,
    }


print("Scene-level Step 3A builder loaded.")

Scene-level Step 3A builder loaded.


In [13]:
# ============================================================
# Run only ONE selected generation mode
#
# RUN_MODE is loaded from pilot_3.config.
# Step 3A automatically processes all available Step 2 files.
# ============================================================

import re

def natural_sort_key(filename):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", filename)
    ]


if RUN_MODE not in {CANONICAL_MODE, DIVERSE_MODE}:
    raise ValueError(
        f"Invalid RUN_MODE: {RUN_MODE}. "
        f"Use '{CANONICAL_MODE}' or '{DIVERSE_MODE}'."
    )

if RUN_MODE == CANONICAL_MODE:
    OUTPUT_DIR = STEP3A_CANONICAL_DIR
elif RUN_MODE == DIVERSE_MODE:
    OUTPUT_DIR = STEP3A_DIVERSE_DIR
else:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

# Clear current mode output directory to avoid mixing old files.
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Selected RUN_MODE:", RUN_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR)

# Automatically detect all Step 2 relation files
expected_input_files = sorted(
    [
        filename
        for filename in os.listdir(STEP2_DIR)
        if filename.endswith("_chunk_relations.json")
    ],
    key=natural_sort_key
)

if not expected_input_files:
    raise FileNotFoundError(
        f"No Step 2 relation files found in directory:\n{STEP2_DIR}\n\n"
        "Expected files ending with '_chunk_relations.json'."
    )

# Derive SCENES from actual Step 2 filenames
SCENES = [
    filename.replace("_chunk_relations.json", "")
    for filename in expected_input_files
]

print("Detected scenes:", SCENES)
print("Input files:", expected_input_files)
print("Number of input files:", len(expected_input_files))


def run_step3a_for_mode(mode: str, output_dir: str) -> List[str]:
    processed = []

    print(f"\n=== Running Step 3A mode: {mode} ===")
    print("Detected scenes:", SCENES)

    for filename in expected_input_files:
        in_path = os.path.join(STEP2_DIR, filename)
        scene_data = load_json(in_path)

        out_data = build_scene_step3a(scene_data, mode=mode)

        out_filename = f"{out_data['scene']}_experiment_A_text_{mode}.json"
        out_path = os.path.join(output_dir, out_filename)

        save_json(out_path, out_data)
        processed.append(out_filename)

        print(
            f"[Saved] {out_filename} | "
            f"scene={out_data['scene']} | "
            f"paragraphs={out_data['num_paragraphs']} | "
            f"pair_targets={out_data['num_pair_targets']} | "
            f"paragraph_relation_counts={out_data['paragraph_relation_counts']}"
        )

    return processed


processed_files = run_step3a_for_mode(
    mode=RUN_MODE,
    output_dir=OUTPUT_DIR,
)

print(f"\nOutput files for mode={RUN_MODE}:")
for f in processed_files:
    print("-", f)

Selected RUN_MODE: diverse
OUTPUT_DIR: /content/pilot_3/data/step3A_diverse_text_preserve_text_direction
Detected scenes: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6']
Input files: ['FloorPlan1_chunk_relations.json', 'FloorPlan2_chunk_relations.json', 'FloorPlan3_chunk_relations.json', 'FloorPlan4_chunk_relations.json', 'FloorPlan5_chunk_relations.json', 'FloorPlan6_chunk_relations.json']
Number of input files: 6

=== Running Step 3A mode: diverse ===
Detected scenes: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6']
[Saved] FloorPlan1_experiment_A_text_diverse.json | scene=FloorPlan1 | paragraphs=24 | pair_targets=2666 | paragraph_relation_counts={'on': 72, 'above': 28, 'contains': 7, 'left_of': 87, 'right_of': 77, 'below': 35, 'supports': 24, 'near': 20, 'in': 4}
[Saved] FloorPlan2_experiment_A_text_diverse.json | scene=FloorPlan2 | paragraphs=20 | pair_targets=2074 | paragraph_relation_counts={'on': 49, 'above

In [14]:
# ============================================================
# Export JSONL for the selected mode only
# Paragraph-level JSONL + probe-pair JSONL
#
# Export only files for config.SCENES.
# ============================================================

def iter_scene_output_files(output_dir: str, mode: str):
    for scene in SCENES:
        filename = f"{scene}_experiment_A_text_{mode}.json"
        path = os.path.join(output_dir, filename)

        if not os.path.exists(path):
            print("[Skip missing output]", filename)
            continue

        yield scene, filename, path


def export_jsonl(output_dir: str, mode: str) -> str:
    """
    Export paragraph-level records.
    Each line is one paragraph record, including text, object_mentions,
    explicit_relations_in_text, and pair_targets.
    """
    jsonl_path = os.path.join(output_dir, f"experiment_A_paragraphs_{mode}_all.jsonl")

    num_written = 0

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for scene, filename, path in iter_scene_output_files(output_dir, mode):
            data = load_json(path)

            for rec in data["paragraphs"]:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                num_written += 1

    print(f"[Paragraph JSONL] mode={mode} | path={jsonl_path} | paragraphs={num_written}")
    return jsonl_path


def export_probe_pairs_jsonl(output_dir: str, mode: str) -> str:
    """
    Export pair-level records for quick inspection.

    This is not the final Step 4 probe dataset.
    Step 4 remains responsible for the final clean direct/inverse probe export.
    """
    out_path = os.path.join(
        output_dir,
        f"experiment_A_probe_pairs_{mode}_labeled_only.jsonl"
    )

    keep_evidence_types = {
        "explicit_direct",
        "explicit_inverse_or_same_sentence_pair",
        "implicit_labeled",
    }

    num_written = 0
    label_counts = Counter()
    evidence_counts = Counter()
    paragraph_counts = Counter()

    with open(out_path, "w", encoding="utf-8") as wf:
        for scene, filename, path in iter_scene_output_files(output_dir, mode):
            data = load_json(path)

            for rec in data["paragraphs"]:
                for pt in rec["pair_targets"]:
                    if pt["pair_evidence_type"] not in keep_evidence_types:
                        continue

                    if pt["relation_label"] == NONE_RELATION_LABEL:
                        continue

                    row = {
                        "paragraph_id": rec["paragraph_id"],
                        "scene": rec["scene"],
                        "chunk_id": rec["chunk_id"],
                        "cluster_id": rec["cluster_id"],
                        "generation_mode": rec["generation_mode"],
                        "paragraph_index_within_cluster": rec.get("paragraph_index_within_cluster"),

                        "text": rec["text"],

                        "subject_uid": pt["subject_uid"],
                        "subject_id": pt["subject_id"],
                        "subject_type": pt["subject_type"],

                        "object_uid": pt["object_uid"],
                        "object_id": pt["object_id"],
                        "object_type": pt["object_type"],

                        "relation_label": pt["relation_label"],
                        "has_relation_label": pt["has_relation_label"],
                        "relation_source": pt["relation_source"],
                        "pair_evidence_type": pt["pair_evidence_type"],
                        "geometry_label": pt.get("geometry_label"),

                        "source_relation_summary": pt.get("source_relation_summary"),
                    }

                    wf.write(json.dumps(row, ensure_ascii=False) + "\n")

                    num_written += 1
                    label_counts[pt["relation_label"]] += 1
                    evidence_counts[pt["pair_evidence_type"]] += 1
                    paragraph_counts[rec["paragraph_id"]] += 1

    print(f"[Probe-pair JSONL] mode={mode} | path={out_path}")
    print("num_probe_pairs:", num_written)
    print("num_paragraphs_with_probe_pairs:", len(paragraph_counts))
    print("label_counts:", dict(label_counts))
    print("evidence_counts:", dict(evidence_counts))

    return out_path


jsonl_path = export_jsonl(
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
)

probe_jsonl_path = export_probe_pairs_jsonl(
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
)

[Paragraph JSONL] mode=diverse | path=/content/pilot_3/data/step3A_diverse_text_preserve_text_direction/experiment_A_paragraphs_diverse_all.jsonl | paragraphs=136
[Probe-pair JSONL] mode=diverse | path=/content/pilot_3/data/step3A_diverse_text_preserve_text_direction/experiment_A_probe_pairs_diverse_labeled_only.jsonl
num_probe_pairs: 8760
num_paragraphs_with_probe_pairs: 136
label_counts: {'right_of': 2427, 'above': 501, 'contains': 75, 'near': 1458, 'on': 648, 'supports': 648, 'below': 501, 'in': 75, 'left_of': 2427}
evidence_counts: {'implicit_labeled': 4978, 'explicit_inverse_or_same_sentence_pair': 1872, 'explicit_direct': 1910}


In [15]:
# ============================================================
# Check paragraph content quality for the selected mode only
# Coverage-aware version
#
# Check only files for config.SCENES.
# ============================================================

OBJECT_ID_LEAK_PATTERN = re.compile(r"[A-Za-z]+?\|[-+0-9\.]+\|[-+0-9\.]+\|[-+0-9\.]+")
ALIAS_PATTERN = re.compile(r"\b[a-z]+(?:_[a-z]+)*_\d+\b")


def split_sentences_simple(text: str):
    parts = [s.strip() for s in text.split(".")]
    return [s for s in parts if s]


def check_paragraph_content(rec):
    issues = []

    text = rec.get("text", "")
    object_mentions = rec.get("object_mentions", [])
    explicit_relations = rec.get("explicit_relations_in_text", [])
    pair_targets = rec.get("pair_targets", [])

    aliases_from_metadata = {
        obj["object_uid"]
        for obj in object_mentions
    }

    aliases_in_text = set(ALIAS_PATTERN.findall(text))
    sentences = split_sentences_simple(text)

    if not text.strip():
        issues.append("empty_text")

    if len(sentences) < MIN_RELATIONS_PER_PARAGRAPH:
        issues.append(f"too_few_sentences: {len(sentences)}")

    if len(text.split()) > 220:
        issues.append(f"text_too_long_words: {len(text.split())}")

    if OBJECT_ID_LEAK_PATTERN.search(text):
        issues.append("object_id_or_coordinate_leakage")

    missing_aliases = sorted(aliases_from_metadata - aliases_in_text)
    if missing_aliases:
        issues.append(f"metadata_aliases_missing_in_text: {missing_aliases}")

    extra_aliases = sorted(aliases_in_text - aliases_from_metadata)
    if extra_aliases:
        issues.append(f"unregistered_aliases_in_text: {extra_aliases}")

    sentence_counts = Counter(sentences)
    duplicate_sentences = [
        sent for sent, count in sentence_counts.items()
        if count > 1
    ]

    if duplicate_sentences:
        issues.append(f"duplicate_sentences: {duplicate_sentences[:3]}")

    if len(explicit_relations) < MIN_RELATIONS_PER_PARAGRAPH:
        issues.append(f"too_few_explicit_relations: {len(explicit_relations)}")

    n = len(object_mentions)
    expected_ordered_pairs = n * (n - 1)

    if INCLUDE_NONE_RELATION_PAIRS and len(pair_targets) != expected_ordered_pairs:
        issues.append(
            f"pair_target_coverage_mismatch: expected {expected_ordered_pairs}, got {len(pair_targets)}"
        )

    diag = rec.get("diagnostics", {})
    coverage = diag.get("object_coverage_ratio")

    if coverage is not None and coverage < 0.5:
        issues.append(f"low_object_coverage_ratio: {coverage:.3f}")

    return issues


def check_output_dir(output_dir: str, mode: str):
    all_issues = []

    coverage_values = []
    object_counts = []
    cluster_object_counts = []
    pair_target_counts = []

    for scene, filename, path in iter_scene_output_files(output_dir, mode):
        data = load_json(path)

        file_issues = []

        for rec in data["paragraphs"]:
            issues = check_paragraph_content(rec)

            diag = rec.get("diagnostics", {})
            coverage = diag.get("object_coverage_ratio")

            if coverage is not None:
                coverage_values.append(coverage)

            object_counts.append(diag.get("num_object_mentions"))
            cluster_object_counts.append(diag.get("source_cluster_num_objects"))
            pair_target_counts.append(diag.get("num_pair_targets"))

            if issues:
                file_issues.append({
                    "paragraph_id": rec["paragraph_id"],
                    "issues": issues,
                    "text": rec["text"],
                    "diagnostics": rec.get("diagnostics", {}),
                })

        print(
            filename,
            "| mode:",
            mode,
            "| paragraphs:",
            data["num_paragraphs"],
            "| paragraphs_with_issues:",
            len(file_issues)
        )

        all_issues.extend(file_issues)

    print(f"\nTotal issues for mode={mode}:", len(all_issues))

    if coverage_values:
        print("\nCoverage summary:")
        print("num paragraphs:", len(coverage_values))
        print("min coverage:", round(min(coverage_values), 3))
        print("max coverage:", round(max(coverage_values), 3))
        print("avg coverage:", round(sum(coverage_values) / len(coverage_values), 3))

    clean_object_counts = [x for x in object_counts if x is not None]
    clean_pair_counts = [x for x in pair_target_counts if x is not None]

    if clean_object_counts:
        print("\nObject mention summary:")
        print("min num_object_mentions:", min(clean_object_counts))
        print("max num_object_mentions:", max(clean_object_counts))
        print("avg num_object_mentions:", round(sum(clean_object_counts) / len(clean_object_counts), 2))

    if clean_pair_counts:
        print("\nPair target summary:")
        print("min num_pair_targets:", min(clean_pair_counts))
        print("max num_pair_targets:", max(clean_pair_counts))
        print("avg num_pair_targets:", round(sum(clean_pair_counts) / len(clean_pair_counts), 2))

    if all_issues:
        from pprint import pprint
        print("\nExample issues:")
        pprint(all_issues[:3])


check_output_dir(
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
)

FloorPlan1_experiment_A_text_diverse.json | mode: diverse | paragraphs: 24 | paragraphs_with_issues: 0
FloorPlan2_experiment_A_text_diverse.json | mode: diverse | paragraphs: 20 | paragraphs_with_issues: 0
FloorPlan3_experiment_A_text_diverse.json | mode: diverse | paragraphs: 24 | paragraphs_with_issues: 0
FloorPlan4_experiment_A_text_diverse.json | mode: diverse | paragraphs: 20 | paragraphs_with_issues: 0
FloorPlan5_experiment_A_text_diverse.json | mode: diverse | paragraphs: 24 | paragraphs_with_issues: 0
FloorPlan6_experiment_A_text_diverse.json | mode: diverse | paragraphs: 24 | paragraphs_with_issues: 0

Total issues for mode=diverse: 0

Coverage summary:
num paragraphs: 136
min coverage: 0.733
max coverage: 1.0
avg coverage: 0.901

Object mention summary:
min num_object_mentions: 5
max num_object_mentions: 12
avg num_object_mentions: 10.15

Pair target summary:
min num_pair_targets: 20
max num_pair_targets: 132
avg num_pair_targets: 97.12


In [16]:
# ============================================================
# Inspect generated paragraph texts for the selected mode only
# Coverage-aware version
#
# Use config.SCENES[0] instead of scanning arbitrary files.
# ============================================================

from pprint import pprint


def load_first_output(output_dir: str, mode: str):
    if not SCENES:
        raise ValueError("SCENES is empty.")

    scene = SCENES[0]
    filename = f"{scene}_experiment_A_text_{mode}.json"
    path = os.path.join(output_dir, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing output file: {path}")

    data = load_json(path)

    return filename, data


sample_file, sample_data = load_first_output(
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
)

print("Sample file:", sample_file)
print("Generation mode:", RUN_MODE)
print("Scene:", sample_data["scene"])
print("Num paragraphs:", sample_data["num_paragraphs"])
print("Num pair targets:", sample_data["num_pair_targets"])

n = min(len(sample_data["paragraphs"]), 5)

for i in range(n):
    rec = sample_data["paragraphs"][i]
    diag = rec["diagnostics"]

    print("\n" + "=" * 100)
    print("Paragraph index:", i)
    print("paragraph_id:", rec["paragraph_id"])
    print("generation_mode:", rec["generation_mode"])

    print("\n[TEXT]")
    print(rec["text"])

    print("\n[coverage]")
    print("source_cluster_num_objects:", diag.get("source_cluster_num_objects"))
    print("num_object_mentions:", diag.get("num_object_mentions"))
    print("object_coverage_ratio:", diag.get("object_coverage_ratio"))
    print("target_object_coverage_ratio:", diag.get("target_object_coverage_ratio"))
    print("max_objects_per_paragraph:", diag.get("max_objects_per_paragraph"))

    print("\n[Explicit relations]")
    for rel in rec["explicit_relations_in_text"]:
        print(
            "-",
            rel["relation"],
            "| template_index:",
            rel["template_index"],
            "| sentence:",
            rel["sentence"]
        )

    print("\n[pair label counts]")
    pprint(diag["pair_label_counts"])

    print("\n[pair evidence counts]")
    pprint(diag["pair_evidence_counts"])

Sample file: FloorPlan1_experiment_A_text_diverse.json
Generation mode: diverse
Scene: FloorPlan1
Num paragraphs: 24
Num pair targets: 2666

Paragraph index: 0
paragraph_id: FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_A_diverse_0
generation_mode: diverse

[TEXT]
Within this local scene, On top of counter_top_1, there is coffee_machine_1. mug_1 is resting on counter_top_1. pot_1 is on counter_top_1. counter_top_1 sits above dish_sponge_1. cabinet_1 includes wine_bottle_1. soap_bottle_1 is on the left side of potato_1. cabinet_2 can be seen to the right of paper_towel_roll_1. Below wine_bottle_1, there is counter_top_1. pot_1 is below cabinet_1. cabinet_2 holds dish_sponge_1. dish_sponge_1 sits to the left of pot_1. dish_sponge_1 is on the left side of potato_1. potato_1 is positioned to the left of pot_1. soap_bottle_1 is positioned to the left of cabinet_1. soap_bottle_1 can be seen to the left of wine_bottle_1. Directly above dish_sponge_1, you can see sink_basin_1.

[coverage]
source_c

In [18]:
# ============================================================
# Zip and download output for the selected mode only
# ============================================================

from google.colab import files

zip_name = f"/content/pilot4_step3A_{RUN_MODE}_text"

zip_path = shutil.make_archive(
    zip_name,
    "zip",
    OUTPUT_DIR,
)

print("Created:", zip_path)

files.download(zip_path)

Created: /content/pilot4_step3A_diverse_text.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>